# BABILong RMM v5p7 — results collection

Scans `runs-rmmv5p7/babilong/**` for finished runs (`config.json` + `all_results.json`) and builds:
- a tidy dataframe of every result,
- **length-extrapolation** pivots (from the `eval_seg<N>_from<SRC>` sweeps),
- **training-curriculum** pivots (per-stage best eval from the `seg<N>_from<M>` runs),
- GitHub-markdown tables + CSV written next to this notebook.

Runs whether launched from `notebooks/` or the repo root. Crashed/in-progress runs (no `all_results.json`) are skipped.

> Note: eval runs are launched without `--learning_rate`, so the eval `config.json` records the runner's default LR, **not** the trained model's. The real LR (and the model identity) is parsed from the `RUN_NAME` in the path.

In [1]:
import json, glob, re
from pathlib import Path
import numpy as np
import pandas as pd

# repo root: works whether this notebook is run from notebooks/ or the repo root
CWD = Path.cwd()
REPO = CWD if (CWD / 'runs-rmmv5p7').exists() else CWD.parent
RUN_GLOB = 'runs-rmmv5p7/babilong/**/config.json'
assert (REPO / 'runs-rmmv5p7').exists(), f'runs-rmmv5p7 not found under {REPO}'
print('repo root:', REPO)

repo root: /home/booydar/Desktop/_projects/2026/opencode-playground/2h100-airi/remote


In [2]:
def collect(repo):
    rows = []
    for cfg in glob.glob(str(repo / RUN_GLOB), recursive=True):
        d = Path(cfg).parent
        res = d / 'all_results.json'
        if not res.exists():
            continue  # crashed / in-progress run
        c = json.load(open(cfg)).get('cli_args', {})
        r = json.load(open(res))
        ep = Path(c.get('exp_path', str(d)))
        leaf, run_name = ep.parts[-2], ep.parts[-3]
        m = re.match(r'(eval_)?seg(\d+)_from(\d+)', leaf)
        # trained LR lives in RUN_NAME (eval runs don't pass --learning_rate)
        lr_m = re.search(r'_lr([0-9.e+-]+)_bs', run_name)
        rows.append(dict(
            task=c.get('task_dataset', '?').split('_')[0],
            write=c.get('write_mode'), read=c.get('read_mode'),
            mem=c.get('num_memory_vectors'), ss=c.get('state_size'),
            lr=lr_m.group(1) if lr_m else '?',
            kind='eval' if leaf.startswith('eval_') else 'train',
            seg=int(m.group(2)) if m else np.nan,
            src=int(m.group(3)) if m else np.nan,
            exact_match=r.get('eval_exact_match'),
            token_acc=r.get('eval_token_accuracy'),
            run_name=run_name, path=str(ep),
        ))
    df = pd.DataFrame(rows)
    if len(df):
        df['variant'] = (df.write + '/' + df.read + ' mem' + df.mem.astype(str)
                         + ' ss' + df.ss.astype(str) + ' lr' + df.lr)
        df = df.sort_values(['task', 'variant', 'kind', 'seg']).reset_index(drop=True)
    return df

df = collect(REPO)
print(f'{len(df)} results  ({(df.kind == "eval").sum()} eval, {(df.kind == "train").sum()} train)')
df[['task', 'variant', 'kind', 'src', 'seg', 'exact_match', 'token_acc']]

29 results  (11 eval, 18 train)


,task,variant,kind,src,seg,exact_match,token_acc
0,qa1,identity/identity mem8 ss32 lr1e-05,eval,32,1,1.000000,1.000000
1,qa1,identity/identity mem8 ss32 lr1e-05,train,0,1,0.568569,0.738153
2,qa1,identity/identity mem8 ss32 lr1e-05,train,1,2,1.000000,1.000000
3,qa1,identity/identity mem8 ss32 lr1e-05,train,2,4,1.000000,1.000000
4,qa1,identity/identity mem8 ss32 lr1e-05,train,4,6,1.000000,1.000000
5,qa1,identity/identity mem8 ss32 lr1e-05,train,6,8,1.000000,1.000000
6,qa1,identity/identity mem8 ss32 lr1e-05,train,8,16,1.000000,1.000000
7,qa1,identity/identity mem8 ss32 lr1e-05,train,16,32,1.000000,1.000000
8,qa1,pool/unpool mem16 ss32 lr1e-04,eval,16,1,1.000000,1.000000
9,qa1,pool/unpool mem16 ss32 lr1e-04,eval,16,2,1.000000,1.000000


In [3]:
def show(pivot):
    """Styler with a red->green heatmap; degrades gracefully if matplotlib is absent."""
    s = pivot.style.format('{:.3f}', na_rep='\u2014')
    try:
        s = s.background_gradient(cmap='RdYlGn', vmin=0.0, vmax=1.0, axis=None)
    except Exception:
        pass
    return s

def to_md(pivot, fmt='{:.3f}'):
    """GitHub-markdown table from a pivot (dependency-free; no tabulate needed)."""
    p = pivot.reset_index()
    cols = [str(c) for c in p.columns]
    lines = ['| ' + ' | '.join(cols) + ' |', '|' + '|'.join('---' for _ in cols) + '|']
    for _, row in p.iterrows():
        cells = []
        for v in row:
            if isinstance(v, float) and pd.isna(v):
                cells.append('')
            elif isinstance(v, float):
                cells.append(fmt.format(v))
            else:
                cells.append(str(v))
        lines.append('| ' + ' | '.join(cells) + ' |')
    return '\n'.join(lines)

## Length-extrapolation eval

Rows = trained variant + source curriculum length (`src`); columns = eval segment count. Values = exact_match.

In [4]:
ev = df[df.kind == 'eval']
eval_em = eval_acc = None
if len(ev):
    eval_em = ev.pivot_table(index=['task', 'variant', 'src'], columns='seg', values='exact_match')
    eval_acc = ev.pivot_table(index=['task', 'variant', 'src'], columns='seg', values='token_acc')
    eval_em.columns = eval_acc.columns = [f'seg{int(c)}' for c in eval_em.columns]
print('exact_match')
show(eval_em) if eval_em is not None else print('(no eval runs yet)')

exact_match


In [5]:
print('token_accuracy')
show(eval_acc) if eval_acc is not None else print('(no eval runs yet)')

token_accuracy


## Training curriculum (per-stage best eval)

Rows = variant; columns = trained stage length; values = exact_match at the end of that stage.

In [6]:
tr = df[df.kind == 'train']
train_em = None
if len(tr):
    train_em = tr.pivot_table(index=['task', 'variant'], columns='seg', values='exact_match')
    train_em.columns = [f'seg{int(c)}' for c in train_em.columns]
show(train_em) if train_em is not None else print('(no training runs yet)')

## Markdown tables + export

Prints copy-pasteable markdown and writes `results_babilong.md` + `results_babilong.csv` next to this notebook.

In [7]:
parts = ['# BABILong RMM v5p7 results\n']
if eval_em is not None:
    parts += ['## Length-extrapolation eval (exact_match)\n', to_md(eval_em), '']
if train_em is not None:
    parts += ['## Training curriculum (exact_match)\n', to_md(train_em), '']
md = '\n'.join(parts)
print(md)

out_md = REPO / 'notebooks' / 'results_babilong.md'
out_csv = REPO / 'notebooks' / 'results_babilong.csv'
out_md.write_text(md)
df.to_csv(out_csv, index=False)
print('\nwrote', out_md, 'and', out_csv)

# BABILong RMM v5p7 results

## Length-extrapolation eval (exact_match)

| task | variant | src | seg1 | seg2 | seg4 | seg8 | seg16 | seg32 | seg64 | seg128 |
|---|---|---|---|---|---|---|---|---|---|---|
| qa1 | identity/identity mem8 ss32 lr1e-05 | 32 | 1.000 |  |  |  |  |  |  |  |
| qa1 | pool/unpool mem16 ss32 lr1e-04 | 16 | 1.000 | 1.000 |  |  |  |  |  |  |
| qa1 | pool/unpool mem16 ss32 lr1e-05 | 32 | 0.568 | 0.547 | 0.529 | 0.508 | 0.514 | 0.518 | 0.515 | 0.520 |

## Training curriculum (exact_match)

| task | variant | seg1 | seg2 | seg4 | seg6 | seg8 | seg16 | seg32 |
|---|---|---|---|---|---|---|---|---|
| qa1 | identity/identity mem8 ss32 lr1e-05 | 0.569 | 1.000 | 1.000 | 1.000 | 1.000 | 1.000 | 1.000 |
| qa1 | pool/unpool mem16 ss32 lr1e-04 | 1.000 | 1.000 | 1.000 | 1.000 | 1.000 |  |  |
| qa1 | pool/unpool mem16 ss32 lr1e-05 | 0.194 | 0.564 | 0.558 | 0.547 | 0.548 | 0.528 |  |


wrote /home/booydar/Desktop/_projects/2026/opencode-playground/2h100-airi/remote/notebooks/resu